# Week 10 - Document Classification (2/2)
**Notebook 2 : Many classifiers, and two ways to represent a document**

**Unstructured Data Analysis (2026-2)** · Professor: Misuk Kim · Teaching Assistant: Hojin Son

Notebook 1 built Naive Bayes and k-NN. Lecture 9-2 also shows, on one slide, five other classifiers and asks
*why are there so many classifiers?* This notebook answers with two experiments, then looks at the last idea of the
Overview chapter: a document as a **vector** or as a **matrix**.

| Lecture chapter | What we do in code |
|---|---|
| 1. Overview, *Why Are There So Many Classifiers?* (p. 11) | the five decision boundaries of the slide on a 2-D toy set |
| 1. Overview, the same question on text | seven classifiers on the 4,000 news articles of Notebook 1, then with less training data |
| 1. Overview, *Vector Space Model vs. Matrix-based Model* (p. 12-13) | one TF-IDF row per article vs one $N \times P$ matrix from BERT |

> **How to use this notebook in Colab**: `File ▸ Save a copy in Drive`, then run the cells from top to bottom with
> `Shift + Enter`. The **🤔 Quick check** notes after an output say what to look for in it.

## &nbsp;0. Setup
The same news sample and the same TF-IDF settings as Notebook 1. Section 3 also uses PyTorch and Hugging Face
`transformers`, both pre-installed in Colab.

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)   # only the MLP convergence messages
%config InlineBackend.figure_format = 'retina'
%matplotlib inline
pd.set_option('display.width', 140)

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/snhzyn/2026-2-Unstructured-Data-Analysis/main/week10-document-classification/data/ag_news_sample.csv"

news = pd.read_csv(DATA_URL)
train = news[news['split'] == 'train'].reset_index(drop=True)
test = news[news['split'] == 'test'].reset_index(drop=True)
CLASSES = ['World', 'Sports', 'Business', 'Sci/Tech']
y_train = train['category'].map(CLASSES.index).to_numpy()
y_test = test['category'].map(CLASSES.index).to_numpy()

tfidf = TfidfVectorizer(stop_words='english', min_df=2)
T_train = tfidf.fit_transform(train['text'])
T_test = tfidf.transform(test['text'])
print('T_train :', T_train.shape, '  T_test :', T_test.shape)

## &nbsp;1. Why are there so many classifiers?
*We cannot guarantee that a single classifier is always better than the others* (p. 11). The slide draws the decision
boundary of five classifiers on the same two-class data. We redraw it on a 2-D toy set (two interleaving half-moons,
400 points), and add k-NN from Notebook 1. Each panel is trained on 300 points and scored on the other 100.

In [ ]:
X2, y2 = make_moons(n_samples=400, noise=0.30, random_state=42)
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=100, random_state=42, stratify=y2)

toy_models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'SVM (Gaussian kernel)': SVC(kernel='rbf'),
    'Neural Network': MLPClassifier(hidden_layer_sizes=(32, 32), max_iter=2000, random_state=42),
    'k-NN (k=15)': KNeighborsClassifier(n_neighbors=15),
}

xx, yy = np.meshgrid(np.linspace(-2, 3, 300), np.linspace(-1.6, 2.1, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(2, 3, figsize=(13, 7.5))
for ax, (name, model) in zip(axes.ravel(), toy_models.items()):
    model.fit(X2_tr, y2_tr)
    zz = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
    ax.scatter(X2_tr[:, 0], X2_tr[:, 1], c=y2_tr, cmap='coolwarm', s=12, edgecolor='k', linewidth=0.3)
    ax.set_title(f'{name}\ntest accuracy {model.score(X2_te, y2_te):.2f}', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

**🤔 Quick check** Six shapes for the same points, as on the slide: a straight line (Logistic Regression), axis-parallel
boxes (Decision Tree), a smoother union of many boxes (Random Forest), and curved boundaries (SVM with a Gaussian kernel,
Neural Network, k-NN). The single tree follows stray points with thin strips and scores lowest. The accuracies are close
(0.87 to 0.93 on 100 test points): with this much noise, even a straight line separates most of the two moons.

## &nbsp;2. The same question on text
### &nbsp;2-1. Seven classifiers on the 4,000 news articles
Every model gets the same TF-IDF rows. Naive Bayes takes the TF-IDF weights in place of counts here (it accepts any
non-negative values), and k-NN uses the best $k$ of Notebook 1.

In [ ]:
def make_models(k=19):
    return {
        'Naive Bayes': MultinomialNB(alpha=1.0),
        f'k-NN (k={k})': KNeighborsClassifier(n_neighbors=k),
        'Logistic Regression': LogisticRegression(max_iter=1000),
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
        'SVM (Gaussian kernel)': SVC(kernel='rbf'),
        'Neural Network': MLPClassifier(hidden_layer_sizes=(64,), max_iter=50, random_state=42),
    }


rows = []
for name, model in make_models().items():
    t0 = time.time(); model.fit(T_train, y_train)
    t1 = time.time(); acc = model.score(T_test, y_test)
    t2 = time.time()
    rows.append({'classifier': name, 'test accuracy': round(acc, 3),
                 'fit (s)': round(t1 - t0, 2), 'predict (s)': round(t2 - t1, 2)})
results = pd.DataFrame(rows).set_index('classifier')
results.sort_values('test accuracy', ascending=False)

**🤔 Quick check** On text, the ranking is not the one of the toy set. The single tree, a few points behind on the moons,
is last by far: one split on one word at a time is a poor fit for 7,982 sparse columns. Logistic Regression, SVM and Naive Bayes finish
within a point of each other, and Naive Bayes gets there in a few milliseconds while the SVM and the Neural Network take
seconds. These are test accuracies, shown to compare: to *choose* a model we would compare on validation data, as with
$k$ in Notebook 1, and use the test set once at the end.

### &nbsp;2-2. Less training data, another ranking
Run the same seven with **25**, **100** and all **1,000** training articles per category (the vocabulary is learned
again each time). k-NN keeps $k = 15$ so that it still makes sense with 100 articles. With 25 articles per
category the vectorizer keeps every word (`min_df=1`): with `min_df=2` only about 300 words would be left. So the
columns differ in the amount of data **and** in `min_df`.

In [ ]:
rng = np.random.RandomState(42)
order = [rng.permutation(np.where(y_train == c)[0]) for c in range(len(CLASSES))]   # one shuffle, reused

table = {}
for n_per_class in [25, 100, 1000]:
    idx = np.concatenate([o[:n_per_class] for o in order])
    vec = TfidfVectorizer(stop_words='english', min_df=2 if n_per_class >= 100 else 1)
    A = vec.fit_transform(train['text'].iloc[idx])
    B = vec.transform(test['text'])
    table[f'{n_per_class} per class'] = {name: round(m.fit(A, y_train[idx]).score(B, y_test), 3)
                                         for name, m in make_models(k=15).items()}

curve = pd.DataFrame(table)
print('best per column :', curve.idxmax().to_dict())
curve

**🤔 Quick check** The winner changes with the amount of data: with 25 or 100 articles per category Naive Bayes is on
top, with 1,000 Logistic Regression and the SVM are. Counting words needs little data; a boundary with thousands of weights
needs more. This is the slide's point in one table: there is no classifier that is always best, so in practice we try
several and keep the one that does best on **validation** data (the same rule as choosing $k$ in Notebook 1).

## &nbsp;3. Vector Space Model vs. Matrix-based Model
*Vector Space Model* (p. 12): *a single document is transformed into a single vector* (DTM, topic models, Doc2Vec ...).
Everything so far, in Notebook 1 and above, used this model.

*Matrix-based Model* (p. 13): *a document is represented as a N by P matrix*, with
*N: the maximum number of words in a single document (ex: 512 for BERT)* and *P: the word embedding dimension (ex: 128)*.
One row per word, in order, and **padding** rows up to $N$.

### &nbsp;3-1. One vector, and what it forgets

In [ ]:
s1 = 'Oracle buys PeopleSoft'
s2 = 'PeopleSoft buys Oracle'

V1, V2 = tfidf.transform([s1, s2])
print('one article -> one vector of shape', V1.shape)
print('non-zero    :', {w: round(float(v), 3) for w, v in zip(tfidf.get_feature_names_out()[V1.indices], V1.data)})
print('same vector for both sentences :', (V1 != V2).nnz == 0)

**🤔 Quick check** One headline, one row of 7,982 numbers, three of them non-zero. The two sentences describe opposite
deals and get the **same** vector: a bag of words keeps which words occur, not their order (the issue of Week 4).

### &nbsp;3-2. One matrix : $N \times P$ from BERT
We use **BERT-Tiny** (`prajjwal1/bert-tiny`, 17 MB), a small BERT whose word embeddings have $P = 128$ dimensions, the
example of p. 13 (BERT-base has 768). The tokenizer cuts the text into tokens, adds `[CLS]` at the start and `[SEP]`
at the end, and pads with `[PAD]` up to $N = 512$. The model returns one 128-dimensional row per position.
We only **look** at the matrix; nothing is trained here.

The first run downloads the model from Hugging Face (a few seconds). A warning about `HF_TOKEN` can be ignored.
If `import transformers` fails with `ModuleNotFoundError` (Colab normally has it), run `!pip install -q transformers`
in a new cell and run this cell again.

In [ ]:
import torch
from transformers import BertTokenizer, BertModel

MODEL = 'prajjwal1/bert-tiny'
tokenizer = BertTokenizer.from_pretrained(MODEL)
bert = BertModel.from_pretrained(MODEL)
bert.eval()

doc = test.loc[0, 'text']
enc = tokenizer(doc, padding='max_length', max_length=512, truncation=True, return_tensors='pt')
with torch.no_grad():
    H = bert(**enc).last_hidden_state[0]           # N x P : one row per token position

tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
n_real = int(enc['attention_mask'].sum())
print(doc, '\n')
print('matrix N x P   :', tuple(H.shape))
print('real tokens    :', n_real, '  padding rows :', 512 - n_real)

A look inside the matrix: the first 12 rows, the end of the article and the start of the padding, and the last two rows.

In [ ]:
view = pd.DataFrame(H[:, :6].numpy().round(2), index=[f'{i:>3} {t}' for i, t in enumerate(tokens)],
                    columns=[f'dim {j}' for j in range(1, 7)])
pd.concat([view.iloc[:12], view.iloc[n_real - 2:n_real + 3], view.iloc[-2:]])

**🤔 Quick check** This is the picture of p. 13: the tokens of the article from top to bottom (*argentina*, then the rest),
`[SEP]`, and then `[PAD]` rows until row 511. Only the first 6 of the 128 columns are shown. The padding rows still hold
numbers; the `attention_mask` (1 for a real token, 0 for padding) tells the model to ignore them.
Rare words are split into pieces marked `##` (*\$20bn* becomes `$`, `20`, `##bn`): BERT's vocabulary has about 30,000
pieces, so almost no word is out of vocabulary (a character that is not in it at all becomes `[UNK]`).

### &nbsp;3-3. The matrix keeps the order

In [ ]:
def bert_matrix(text):
    e = tokenizer(text, return_tensors='pt')                 # no padding: just the real tokens
    with torch.no_grad():
        return bert(**e).last_hidden_state[0], tokenizer.convert_ids_to_tokens(e['input_ids'][0])


H1, t1 = bert_matrix(s1)
H2, t2 = bert_matrix(s2)
print(s1, '->', t1)
print(s2, '->', t2)
print('same matrix :', torch.allclose(H1, H2))

cos = torch.nn.functional.cosine_similarity
oracle_1, oracle_2 = H1[t1.index('oracle')], H2[t2.index('oracle')]
print('cosine of the two "oracle" rows :', round(float(cos(oracle_1, oracle_2, dim=0)), 3))

**🤔 Quick check** Same words, different matrices: the rows come in a different order, and even the row of *oracle* itself
changes (cosine below 1), because BERT computes each row from the whole sentence. A word gets a different vector in a
different context, which Word2Vec of Week 6 could not do.

A classifier that reads such a matrix directly (a CNN, an RNN, or BERT itself fine-tuned on labeled articles) is outside
lecture 9-2. The classifiers of this lecture all take **one vector per document**.

---
### ✅ What we practiced

| Lecture | In code |
|---|---|
| *Why Are There So Many Classifiers?* (p. 11) | six decision boundaries on the same toy data; seven classifiers on the same news |
| no classifier is always best | the ranking changes between toy data and text, and with the amount of training data |
| *Vector Space Model* (p. 12) | one TF-IDF row per article; word order is lost |
| *Matrix-based Model* (p. 13) | BERT-Tiny turns an article into a $512 \times 128$ matrix with padding rows |

**Assignment 5** practices the code of Notebook 1: Naive Bayes and k-NN, step by step.